# dagsampler — Quickstart

A minimal walk-through of the public API:

1. Build a config dict describing the DAG and node mechanisms.
2. Pass it to `CausalDataGenerator(...).simulate()`.
3. Inspect the returned `data`, `dag`, and `parametrization`.

See the [documentation](https://averinpa.github.io/dagsampler/) for the full reference.

In [1]:
from dagsampler import CausalDataGenerator

## 1. Build a config

We'll define a simple v-structure (collider): `X -> Z` and `Y -> Z`.
All three nodes are continuous; `X` and `Y` are exogenous Gaussians, `Z` is
a linear combination plus additive Gaussian noise.

In [2]:
config = {
    "simulation_params": {"n_samples": 500, "seed": 42},
    "graph_params": {
        "type": "custom",
        "nodes": ["X", "Y", "Z"],
        "edges": [["X", "Z"], ["Y", "Z"]],
    },
    "node_params": {
        "X": {"type": "continuous", "distribution": {"name": "gaussian", "mean": 0, "std": 1}},
        "Y": {"type": "continuous", "distribution": {"name": "gaussian", "mean": 0, "std": 1}},
        "Z": {
            "type": "continuous",
            "functional_form": {"name": "linear", "weights": {"X": 1.2, "Y": -0.7}},
            "noise_model": {"name": "additive", "dist": "gaussian", "std": 0.3},
        },
    },
}

## 2. Simulate

`simulate()` returns a dict with `data`, `dag`, `parametrization`, and
(optionally) `ci_oracle`.

In [3]:
result = CausalDataGenerator(config).simulate()
list(result.keys())

['data', 'parametrization', 'dag']

## 3. Inspect outputs

### Data — a pandas DataFrame

In [4]:
result["data"].head()

,X,Y,Z
0,0.244230,0.553531,0.086986
1,0.678178,0.846116,0.588565
2,-0.585529,1.803438,-1.967209
3,-0.908673,0.175773,-1.240507
4,-1.991838,0.469270,-3.035840


In [5]:
result["data"].describe()

,X,Y,Z
count,500.000000,500.000000,500.000000
mean,0.003410,-0.045966,0.046365
std,0.987813,1.016777,1.402966
min,-2.772900,-3.378306,-3.756199
25%,-0.683268,-0.655424,-0.873474
50%,0.031373,-0.064866,0.046310
75%,0.668475,0.575473,0.980920
max,2.939544,3.147821,4.682744


### DAG — a `networkx.DiGraph`

In [6]:
list(result["dag"].edges())

[('X', 'Z'), ('Y', 'Z')]

### Parametrization — a deep copy of the config with every sampled value filled in

This is what you would persist to JSON to reproduce the exact DGP.

In [7]:
result["parametrization"]["node_params"]["Z"]

{'type': 'continuous',
 'functional_form': {'name': 'linear', 'weights': {'X': 1.2, 'Y': -0.7}},
 'noise_model': {'name': 'additive', 'dist': 'gaussian', 'std': 0.3},
 'post_transform': {'name': None}}

In [8]:
result["parametrization"]["simulation_params"]

{'n_samples': 500, 'seed': 42, 'seed_structure': 42, 'seed_data': 43}

## 4. Reproducibility

Same config → identical data. The simulator uses two independent random streams
(`seed_structure` for the DGP, `seed_data` for the per-row draws); when only
`seed` is given, `seed_data` is derived as `seed + 1`.

In [9]:
again = CausalDataGenerator(config).simulate()
(result["data"] == again["data"]).all().all()

np.True_

## 5. Optional: CI oracle (ground truth from d-separation)

Useful for benchmarking conditional-independence tests.

In [10]:
config_with_oracle = {
    **config,
    "simulation_params": {**config["simulation_params"], "store_ci_oracle": True, "ci_oracle_max_cond_set": 1},
}
oracle = CausalDataGenerator(config_with_oracle).simulate()["ci_oracle"]
oracle[:5]

[{'x': 'X', 'y': 'Y', 'conditioning_set': [], 'is_independent': True},
 {'x': 'X', 'y': 'Y', 'conditioning_set': ['Z'], 'is_independent': False},
 {'x': 'X', 'y': 'Z', 'conditioning_set': [], 'is_independent': False},
 {'x': 'X', 'y': 'Z', 'conditioning_set': ['Y'], 'is_independent': False},
 {'x': 'Y', 'y': 'Z', 'conditioning_set': [], 'is_independent': False}]

## Next steps

- See [`02_templates_and_features.ipynb`](./02_templates_and_features.ipynb) for template DAGs and the new structural forms (`sigmoid`, `cos`, `sin`, `post_transform`).
- Full configuration reference: [Configuration Examples](https://averinpa.github.io/dagsampler/config_examples.html)
- Mathematical formulations: [Model Formulations](https://averinpa.github.io/dagsampler/formulations.html)